# ChangeGuardian AI
### LangGraph · NetworkX Graph RAG · FAISS Vector RAG · Vectorless Rules · Local LLM (vLLM/AMD)

**Architecture:**
```
Change Request
     
[1] Intake Agent           extract service, old/new value, extra params
    
[2] Scenario Router      classify: framework_upgrade | resource_change | db_schema |
                                   api_contract | shared_dependency | event_schema
    
[3] Graph Impact Agent     NetworkX: find all affected services/DBs/APIs/Kafka
     
[4] Hybrid RAG Agent       FAISS vector search (similar incidents)
                          + Vectorless rule lookup (exact constraint checks)
     
[5] Memory Graph Agent    check prior deployment outcomes for same service
  
[6] Risk & Rollout Agent   deterministic scoring (0-100) + rollout decision
   
[7] LLM Explanation Agent Local Qwen via vLLM (OpenAI-compatible API on port 8000)
                           generates: plain-english explanation + remediation steps
    
   Report
```

**Why AMD + vLLM?** The local LLM loads entirely into system RAM (or AMD GPU via ROCm). No cloud API. No data leaves the machine. Runs on AMD EPYC / Ryzen with large RAM or AMD Radeon GPU. vLLM provides fast OpenAI-compatible inference.

In [ ]:
# Cell 1 — Install dependencies
# numpy 2.x is the resident version on this machine.
# Fix: upgrade pyarrow (16+) and scikit-learn (1.5+) to numpy-2.x compatible builds.

!pip install "pyarrow>=16.0.0" --upgrade -q
!pip install "scikit-learn>=1.5.0" --upgrade -q
!pip install langgraph networkx faiss-cpu sentence-transformers gradio psutil requests -q

import numpy, pyarrow, sklearn
print(f"numpy        : {numpy.__version__}")
print(f"pyarrow      : {pyarrow.__version__}")
print(f"scikit-learn : {sklearn.__version__}")
print("All dependencies OK — proceed to Cell 2.")

# ── vLLM setup (for local LLM on AMD hardware with OpenAI-compatible API) ──────────────────────
# Install vLLM: pip install vllm
#
# Supported Models (auto-selected by available RAM in Cell 2):
#   • qwen2.5:3b        - Requires: 16 GB RAM (smallest, fastest)
#   • qwen2.5:7b        - Requires: 32 GB RAM
#   • qwen2.5:14b       - Requires: 64 GB RAM
#   • qwen3:30b-a3b     - Requires: 128 GB RAM (recommended for best quality)
#   • llama3.1:70b      - Requires: 256 GB RAM (largest, slowest but highest quality)
#
# To start vLLM with your model, run in a terminal:
#   python -m vllm.entrypoints.openai.api_server \
#     --model <huggingface-model-name> \
#     --port 8000 \
#     --tensor-parallel-size 1
#
# Example for Qwen 30B A3B:
#   python -m vllm.entrypoints.openai.api_server \
#     --model Qwen/Qwen-30B-A3B-Instruct \
#     --port 8000
#
# The notebook works WITHOUT vLLM — rule-based fallback activates automatically.

In [ ]:
# Cell 2 – AMD System Info + vLLM Setup (Optimized for 14B Models)
import psutil, platform, os, requests, json, time
import subprocess
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import hashlib

# ── System info (AMD RAM / CPU) ─────────────────────────────────────────────────────────────────
ram_gb      = psutil.virtual_memory().total / (1024 ** 3)
cpu_cores   = psutil.cpu_count(logical=True)
cpu_phys    = psutil.cpu_count(logical=False)
cpu_name    = platform.processor() or "Unknown CPU"

print("=" * 60)
print("  AMD SYSTEM PROFILE")
print("=" * 60)
print(f"  CPU          : {cpu_name}")
print(f"  Physical cores: {cpu_phys}    Logical cores: {cpu_cores}")
print(f"  Total RAM    : {ram_gb:.1f} GB")
print(f"  Available RAM: {psutil.virtual_memory().available / (1024**3):.1f} GB")
print("=" * 60)

# ── vLLM Model Specifications (reference database) ───────────────────────────────────────────────
MODEL_SPECS = {
    "qwen2.5-3b": {"vram_gb": 6, "ram_gb": 16, "context_window": 32768, "temperature": 0.2, "timeout": 90, "reasoning_depth": "basic"},
    "qwen2.5-7b": {"vram_gb": 12, "ram_gb": 32, "context_window": 32768, "temperature": 0.3, "timeout": 120, "reasoning_depth": "intermediate"},
    "qwen2.5-14b": {"vram_gb": 20, "ram_gb": 64, "context_window": 32768, "temperature": 0.3, "timeout": 150, "reasoning_depth": "deep"},
    "qwen-30b": {"vram_gb": 35, "ram_gb": 128, "context_window": 128000, "temperature": 0.2, "timeout": 180, "reasoning_depth": "expert"},
    "llama-70b": {"vram_gb": 60, "ram_gb": 256, "context_window": 131072, "temperature": 0.2, "timeout": 200, "reasoning_depth": "expert"},
}

# ── HTTP Session with Connection Pooling (OPTIMIZATION #1) ───────────────────────────────────────
def create_pooled_session():
    """Create HTTP session with connection pooling for vLLM API."""
    session = requests.Session()
    retry_strategy = Retry(total=2, backoff_factor=0.1, status_forcelist=[429, 500, 502, 503])
    adapter = HTTPAdapter(max_retries=retry_strategy, pool_connections=10, pool_maxsize=10)
    session.mount("http://", adapter)
    return session

VLLM_SESSION = create_pooled_session()

# ── Response Caching (OPTIMIZATION #4) ──────────────────────────────────────────────────────────
class ResponseCache:
    def __init__(self, max_size=100):
        self.cache = {}
        self.max_size = max_size
    
    def get_key(self, prompt: str, system: str) -> str:
        """Generate cache key from prompt+system."""
        combined = f"{system}||{prompt}"
        return hashlib.md5(combined.encode()).hexdigest()
    
    def get(self, prompt: str, system: str):
        """Get cached response if exists."""
        key = self.get_key(prompt, system)
        return self.cache.get(key)
    
    def set(self, prompt: str, system: str, response: str):
        """Cache response (LRU eviction if full)."""
        if len(self.cache) >= self.max_size:
            self.cache.pop(next(iter(self.cache)))
        key = self.get_key(prompt, system)
        self.cache[key] = response
    
    def clear(self):
        """Clear all cached responses."""
        self.cache.clear()
    
    def stats(self):
        """Return cache hit statistics."""
        return f"Cache: {len(self.cache)}/{self.max_size} items"

response_cache = ResponseCache()

# ── Auto-Detect Available Models from vLLM API ──────────────────────────────────────────────────
def get_available_models() -> list:
    """Query vLLM /v1/models endpoint to get available models."""
    try:
        resp = VLLM_SESSION.get("http://localhost:8000/v1/models", timeout=3)
        if resp.status_code == 200:
            data = resp.json()
            return [m.get("id") for m in data.get("data", [])]
    except Exception as e:
        print(f"  ⚠️  Could not fetch models: {e}")
    return []

def match_model_spec(model_name: str) -> str:
    """Match a model name to a spec key."""
    model_lower = model_name.lower()
    for spec_key in MODEL_SPECS:
        if spec_key in model_lower:
            return spec_key
    return None

def select_best_model(available_models: list, ram_gb: float, preferred: str = None) -> tuple:
    """Select best model from available ones based on RAM."""
    if not available_models:
        print("  ⚠️  No models available from vLLM!")
        return None, None, None
    
    print(f"\n  Available models: {available_models}")
    
    if preferred:
        for model in available_models:
            if preferred.lower() in model.lower():
                spec_key = match_model_spec(model)
                if spec_key and ram_gb >= MODEL_SPECS[spec_key]["ram_gb"]:
                    return model, spec_key, MODEL_SPECS[spec_key]
    
    size_order = ["30b", "70b", "14b", "7b", "3b"]
    sorted_models = sorted(available_models, key=lambda x: min((i for i, s in enumerate(size_order) if s in x.lower()), default=999))
    
    for model in sorted_models:
        spec_key = match_model_spec(model)
        if spec_key and ram_gb >= MODEL_SPECS[spec_key]["ram_gb"]:
            return model, spec_key, MODEL_SPECS[spec_key]
    
    smallest = sorted_models[-1]
    spec_key = match_model_spec(smallest)
    return smallest, spec_key, (MODEL_SPECS[spec_key] if spec_key else {})

# ── Detect and Select Model ─────────────────────────────────────────────────────────────────────
VLLM_URL = "http://localhost:8000/v1/completions"
VLLM_TOKENIZE_URL = "http://localhost:8000/tokenize"

print("\n  Detecting available models...")
available_models = get_available_models()

if available_models:
    VLLM_MODEL, MODEL_SPEC_KEY, MODEL_CONFIG = select_best_model(available_models, ram_gb)
    if not MODEL_CONFIG:
        MODEL_CONFIG = {"vram_gb": 20, "ram_gb": 64, "context_window": 32768, "temperature": 0.3, "timeout": 150, "reasoning_depth": "unknown"}
    OLLAMA_MODEL = MODEL_SPEC_KEY or VLLM_MODEL
else:
    VLLM_MODEL = None
    MODEL_CONFIG = {}
    OLLAMA_MODEL = "offline"

# ── Token Counter (OPTIMIZATION: reuse session) ─────────────────────────────────────────────────
def count_tokens(text: str) -> int:
    """Count tokens using vLLM's /tokenize endpoint."""
    try:
        resp = VLLM_SESSION.post(VLLM_TOKENIZE_URL, json={"prompt": text}, timeout=5)
        if resp.status_code == 200:
            return len(resp.json().get("tokens", []))
    except Exception:
        pass
    return 0

# ── GPU/VRAM Detection ──────────────────────────────────────────────────────────────────────────
def get_gpu_vram_usage() -> dict:
    """Get GPU VRAM usage from nvidia-smi or rocm-smi."""
    try:
        result = subprocess.run(['nvidia-smi', '--query-gpu=memory.used,memory.total', '--format=csv,nounits,noheader'],
                              capture_output=True, text=True, timeout=2)
        if result.returncode == 0 and result.stdout.strip():
            used, total = map(float, result.stdout.strip().split('\n')[0].split(','))
            return {"used_gb": used / 1024, "total_gb": total / 1024, "gpu_type": "NVIDIA"}
    except: pass
    
    try:
        result = subprocess.run(['rocm-smi', '--showmeminfo'], capture_output=True, text=True, timeout=2)
        if result.returncode == 0 and result.stdout.strip():
            return {"used_gb": 0, "total_gb": 0, "gpu_type": "AMD"}
    except: pass
    
    return {"used_gb": 0, "total_gb": 0, "gpu_type": "None"}

# ── Metrics Tracking (Optimized) ────────────────────────────────────────────────────────────────
class MetricsTracker:
    def __init__(self, sample_rate=1.0):
        self.sample_rate = sample_rate
        self.reset()
    
    def reset(self):
        self.llm_calls = 0
        self.sampled_calls = 0
        self.total_tokens = 0
        self.total_input_tokens = 0
        self.total_output_tokens = 0
        self.llm_time = 0.0
        self.peak_vram = 0.0
        self.peak_gpu_vram = 0.0
        self.start_ram = psutil.virtual_memory().used / (1024**3)
    
    def get_system_metrics(self):
        ram_used = psutil.virtual_memory().used / (1024**3)
        ram_percent = psutil.virtual_memory().percent
        gpu_info = get_gpu_vram_usage()
        return {
            "ram_used_gb": round(ram_used, 2),
            "ram_percent": round(ram_percent, 1),
            "ram_delta_gb": round(ram_used - self.start_ram, 2),
            "gpu_vram_used_gb": round(gpu_info.get("used_gb", 0), 2),
            "gpu_vram_total_gb": round(gpu_info.get("total_gb", 0), 2),
            "gpu_type": gpu_info.get("gpu_type", "None"),
        }
    
    def log_llm_call(self, elapsed_time, input_text, output_text):
        self.llm_calls += 1
        should_sample = (self.llm_calls % int(1/self.sample_rate) == 0) if self.sample_rate < 1 else True
        
        if should_sample:
            self.sampled_calls += 1
            input_tokens = count_tokens(input_text)
            output_tokens = count_tokens(output_text)
            self.total_input_tokens += input_tokens
            self.total_output_tokens += output_tokens
            self.total_tokens += input_tokens + output_tokens
            
            metrics = self.get_system_metrics()
            self.peak_vram = max(self.peak_vram, metrics["ram_used_gb"])
            self.peak_gpu_vram = max(self.peak_gpu_vram, metrics.get("gpu_vram_used_gb", 0))
            print(f"\n  [LLM Call #{self.llm_calls}]")
            print(f"    Time: {elapsed_time:.2f}s | RAM: {metrics['ram_used_gb']:.1f}GB ({metrics['ram_percent']:.1f}%)")
            if input_tokens > 0 or output_tokens > 0:
                print(f"    Tokens: Input={input_tokens}, Output={output_tokens}")
        else:
            self.llm_time += elapsed_time
    
    def print_summary(self):
        print("\n" + "="*60)
        print("  METRICS SUMMARY (14B OPTIMIZED)")
        print("="*60)
        print(f"  LLM Calls        : {self.llm_calls}")
        print(f"  Sampled          : {self.sampled_calls}/{self.llm_calls} ({self.sampled_calls/max(1,self.llm_calls)*100:.0f}%)")
        print(f"  Total LLM Time   : {self.llm_time:.2f}s")
        if self.llm_calls > 0: print(f"  Avg Time/Call    : {self.llm_time/self.llm_calls:.2f}s")
        print(f"  Total Tokens     : {self.total_tokens} (Input: {self.total_input_tokens}, Output: {self.total_output_tokens})")
        print(f"  Peak RAM Used    : {self.peak_vram:.1f}GB / {ram_gb:.1f}GB ({self.peak_vram/ram_gb*100:.1f}%)")
        if self.peak_gpu_vram > 0:
            print(f"  Peak GPU VRAM    : {self.peak_gpu_vram:.1f}GB")
        print(f"  {response_cache.stats()}")
        print("="*60)

metrics = MetricsTracker(sample_rate=1.0)

# ── Test vLLM connectivity ──────────────────────────────────────────────────────────────────────
def _vllm_available() -> bool:
    try:
        r = VLLM_SESSION.get("http://localhost:8000/v1/models", timeout=3)
        return r.status_code == 200
    except Exception:
        return False

OLLAMA_READY = _vllm_available()
print(f"\n  Selected Model   : {VLLM_MODEL}")
print(f"  Model Spec       : {MODEL_SPEC_KEY}")
print(f"  Context Window   : {MODEL_CONFIG.get('context_window', 'unknown'):,} tokens")
print(f"  Reasoning Depth  : {MODEL_CONFIG.get('reasoning_depth', 'unknown').upper()}")
print(f"  vLLM API Status  : {'RUNNING ✓' if OLLAMA_READY else 'NOT FOUND'}")
print(f"  Optimizations    : ✓ Connection pooling, ✓ Response caching, ✓ Metrics sampling")

gpu_check = get_gpu_vram_usage()
print(f"  GPU Available    : {gpu_check.get('gpu_type', 'None')}")

# ── LLM call wrapper with caching ───────────────────────────────────────────────────────────────
def call_llm(prompt: str, system: str = "", use_cache: bool = True) -> str:
    if not OLLAMA_READY or not VLLM_MODEL:
        return "[LLM offline]"
    
    if use_cache:
        cached = response_cache.get(prompt, system)
        if cached:
            return cached
    
    try:
        full_prompt = f"{system}\n\n{prompt}" if system else prompt
        t0 = time.time()
        resp = VLLM_SESSION.post(
            VLLM_URL,
            json={
                "model": VLLM_MODEL,
                "prompt": full_prompt,
                "temperature": MODEL_CONFIG.get("temperature", 0.3),
                "max_tokens": 500,
                "stream": False,
            },
            timeout=MODEL_CONFIG.get("timeout", 150)
        )
        elapsed = time.time() - t0
        data = resp.json()
        
        if "choices" in data and len(data["choices"]) > 0:
            output = data["choices"][0].get("text", "").strip()
            metrics.log_llm_call(elapsed, full_prompt, output)
            if use_cache:
                response_cache.set(prompt, system, output)
            return output
        return "[vLLM error]"
    except Exception as e:
        return f"[Error: {str(e)[:50]}]"

if OLLAMA_READY:
    print("\nWarming up LLM...")
    metrics.reset()
    call_llm("Ready")
    print(f"  LLM ready. Peak RAM: {metrics.peak_vram:.1f}GB")

In [3]:
# Cell 3 â€” Enterprise Data (realistic production inventory)

services = {
    "payment-service":   {"java": "11", "spring_boot": "2.7", "memory_limit_gb": 2,   "peak_memory_gb": 1.6, "restarts_30d": 3,  "criticality": "critical", "db": "payment-db",   "team": "payments"},
    "checkout-service":  {"java": "11", "spring_boot": "2.7", "memory_limit_gb": 2,   "peak_memory_gb": 1.8, "restarts_30d": 8,  "criticality": "critical", "db": "checkout-db",  "team": "checkout"},
    "order-service":     {"java": "17", "spring_boot": "3.1", "memory_limit_gb": 1,   "peak_memory_gb": 0.6, "restarts_30d": 1,  "criticality": "high",     "db": "order-db",    "team": "orders"},
    "user-service":      {"java": "17", "spring_boot": "3.0", "memory_limit_gb": 1,   "peak_memory_gb": 0.4, "restarts_30d": 0,  "criticality": "medium",   "db": "user-db",     "team": "platform"},
    "notification-svc":  {"java": "11", "spring_boot": "2.7", "memory_limit_gb": 0.5, "peak_memory_gb": 0.3, "restarts_30d": 2,  "criticality": "low",      "db": None,          "team": "platform"},
    "legacy-auth-svc":   {"java": "8",  "spring_boot": "2.3", "memory_limit_gb": 1,   "peak_memory_gb": 0.7, "restarts_30d": 5,  "criticality": "high",     "db": "auth-db",     "team": "security"},
    "inventory-service": {"java": "11", "spring_boot": "2.7", "memory_limit_gb": 1,   "peak_memory_gb": 0.5, "restarts_30d": 1,  "criticality": "medium",   "db": "inventory-db","team": "warehouse"},
}

libraries = {
    "legacy-auth-client": {
        "current": "1.4.0", "breaking_version": "2.0.0",
        "used_by": ["payment-service", "checkout-service", "order-service", "user-service", "legacy-auth-svc"],
    },
    "common-utils": {
        "current": "3.1.0", "breaking_version": None,
        "used_by": ["payment-service", "checkout-service", "order-service"],
    },
}

api_consumers = {
    "payment-api": {
        "provider": "payment-service",
        "consumers": ["checkout-service", "order-service", "user-service"],
        "fields": ["customer_id", "amount", "currency", "status"],
    },
    "order-api": {
        "provider": "order-service",
        "consumers": ["notification-svc", "inventory-service"],
        "fields": ["order_id", "customerId", "items", "total"],
    },
}

database_users = {
    "payment-db":   {"shared_by": ["payment-service"],                    "has_nullable_cols": ["transaction.amount"]},
    "checkout-db":  {"shared_by": ["checkout-service"],                   "has_nullable_cols": []},
    "order-db":     {"shared_by": ["order-service", "inventory-service"], "has_nullable_cols": ["order.notes"]},
    "auth-db":      {"shared_by": ["legacy-auth-svc", "user-service"],    "has_nullable_cols": ["user.phone"]},
    "user-db":      {"shared_by": ["user-service"],                       "has_nullable_cols": ["user.middle_name"]},
    "inventory-db": {"shared_by": ["inventory-service"],                  "has_nullable_cols": []},
}

event_consumers = {
    "order-created": {
        "producer": "order-service",
        "fields": ["orderId", "customerId", "customerEmail", "items", "total"],
        "consumers": {
            "notification-svc":  ["customerId", "customerEmail"],
            "inventory-service": ["orderId", "items"],
        },
    },
    "payment-settled": {
        "producer": "payment-service",
        "fields": ["paymentId", "orderId", "amount"],
        "consumers": {"order-service": ["paymentId", "orderId"]},
    },
}

compatibility_rules = {
    "spring_boot_java": {"3.x": ">=17", "2.7.x": ">=11", "2.3.x": ">=8"},
    "spring_boot_breaking": {
        "2.7_to_3.x": [
            "javax -> jakarta namespace migration required",
            "Removed deprecated Spring APIs",
            "spring.factories auto-configuration format removed",
        ]
    },
    "memory_safety_buffer_pct": 0.20,
    "high_restart_threshold": 5,
}

incident_docs = [
    {"id": "INC-001", "service": "checkout-service", "severity": "P1",
     "title": "OOMKilled after memory reduction",
     "root_cause": "Memory limit set below peak usage during Black Friday. Pod restarted 14 times.",
     "lesson": "Never reduce memory limit below 120% of observed peak."},
    {"id": "INC-002", "service": "payment-service", "severity": "P1",
     "title": "Spring Boot 3 migration broke javax imports",
     "root_cause": "javax.persistence not migrated to jakarta.persistence. 100% of payment pods crashed.",
     "lesson": "Run javax->jakarta migration script before upgrading to Spring Boot 3."},
    {"id": "INC-003", "service": "order-service", "severity": "P2",
     "title": "NOT NULL constraint caused bulk insert failure",
     "root_cause": "Added NOT NULL constraint without backfilling existing NULL rows. Deployment blocked 4 hours.",
     "lesson": "Backfill NULLs before adding NOT NULL constraints."},
    {"id": "INC-004", "service": "user-service", "severity": "P2",
     "title": "API field rename broke checkout",
     "root_cause": "Renamed customer_id to customerId without versioning. Checkout returned 500s.",
     "lesson": "Never rename API fields without a version bump or backward-compatible alias."},
    {"id": "INC-005", "service": "legacy-auth-svc", "severity": "P1",
     "title": "legacy-auth-client 2.0 removed token refresh method",
     "root_cause": "Upgraded shared library without checking breaking changes. All services using old token refresh API broke.",
     "lesson": "Check breaking changes in shared library changelog before bulk upgrade."},
    {"id": "INC-006", "service": "notification-svc", "severity": "P2",
     "title": "Missing customerEmail field in Kafka event",
     "root_cause": "customerEmail removed from order-created event. Notification service could not send emails.",
     "lesson": "Never remove event fields consumed by downstream services without a migration period."},
]

memory_graph = [
    {"deployment": "DEP-101", "service": "payment-service",  "change": "Spring Boot 2.5 to 2.7",        "outcome": "success", "lessons": ["Ran full regression suite", "Deployed on low-traffic Sunday"]},
    {"deployment": "DEP-102", "service": "checkout-service", "change": "memory 2GB to 1.5GB",           "outcome": "failed",  "lessons": ["OOMKilled after 2 hours", "Rolled back immediately"]},
    {"deployment": "DEP-103", "service": "order-service",    "change": "ADD NOT NULL order.notes",      "outcome": "failed",  "lessons": ["Forgot to backfill NULLs", "4 hour P2 incident"]},
    {"deployment": "DEP-104", "service": "user-service",     "change": "API field rename user_name",    "outcome": "failed",  "lessons": ["Broke 2 downstream consumers", "Emergency rollback deployed"]},
    {"deployment": "DEP-105", "service": "legacy-auth-svc",  "change": "legacy-auth-client 1.3 to 1.4","outcome": "success", "lessons": ["Minor version â€” no breaking changes", "Tested in staging 1 week"]},
]

print(f"Data loaded: {len(services)} services | {len(libraries)} libs | {len(incident_docs)} incidents | {len(memory_graph)} past deployments")

Data loaded: 7 services | 2 libs | 6 incidents | 5 past deployments


In [4]:
# Cell 4 â€” Graph RAG: NetworkX dependency graph
import networkx as nx

G = nx.DiGraph()

for svc, info in services.items():
    G.add_node(svc, node_type="service", criticality=info["criticality"])

for db, info in database_users.items():
    G.add_node(db, node_type="database")
    for svc in info["shared_by"]:
        G.add_edge(svc, db, rel="USES_DB")

for api, info in api_consumers.items():
    G.add_node(api, node_type="api")
    G.add_edge(info["provider"], api, rel="PROVIDES_API")
    for consumer in info["consumers"]:
        G.add_edge(consumer, api, rel="CONSUMES_API")
        G.add_edge(consumer, info["provider"], rel="CALLS")

for event, info in event_consumers.items():
    G.add_node(event, node_type="kafka_event")
    G.add_edge(info["producer"], event, rel="PRODUCES_EVENT")
    for consumer in info["consumers"]:
        G.add_edge(consumer, event, rel="CONSUMES_EVENT")

for lib, info in libraries.items():
    G.add_node(lib, node_type="library")
    for svc in info["used_by"]:
        G.add_edge(svc, lib, rel="DEPENDS_ON")


def get_affected_services(service_name: str) -> list:
    if service_name not in G:
        return []
    affected = set()
    for node in nx.descendants(G, service_name):
        if G.nodes[node].get("node_type") == "service":
            affected.add(node)
    for node in nx.ancestors(G, service_name):
        if G.nodes[node].get("node_type") == "service":
            affected.add(node)
    affected.discard(service_name)
    return sorted(affected)


print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"payment-service affected: {get_affected_services('payment-service')}")

Graph: 19 nodes, 33 edges
payment-service affected: ['checkout-service', 'inventory-service', 'notification-svc', 'order-service', 'user-service']


In [ ]:
# Cell 5 – Vector RAG: FAISS + SentenceTransformers (lazy-loaded, runs on AMD CPU)
# NOTE: Changed to lazy-loading to avoid blocking on initial import.
# The embedding model only loads when search_incidents() is first called.
import numpy as np, faiss
from sentence_transformers import SentenceTransformer

_embedder = None
_faiss_index = None

def _init_faiss():
    """Initialize FAISS index and embedding model on first use."""
    global _embedder, _faiss_index
    if _embedder is None:
        print(f"Loading embedding model on {cpu_cores} CPU threads (lazy-loaded)...")
        _embedder = SentenceTransformer("all-MiniLM-L6-v2")

        corpus = [
            f"{d['title']}. {d['root_cause']} Lesson: {d['lesson']}"
            for d in incident_docs
        ]

        vecs = _embedder.encode(corpus, convert_to_numpy=True, show_progress_bar=False).astype("float32")
        faiss.normalize_L2(vecs)

        _faiss_index = faiss.IndexFlatIP(vecs.shape[1])
        _faiss_index.add(vecs)
        print(f"FAISS index ready: {_faiss_index.ntotal} vectors | dim={vecs.shape[1]}")


def search_incidents(query: str, k: int = 3) -> list:
    """Search for similar incidents using FAISS vector similarity."""
    _init_faiss()  # Initialize on first call only
    qv = _embedder.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(qv)
    scores, indices = _faiss_index.search(qv, min(k, len(incident_docs)))
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx >= 0:
            doc = incident_docs[idx].copy()
            doc["sim"] = round(float(score), 3)
            results.append(doc)
    return results


print("FAISS vector RAG: lazy-loaded (will initialize on first search)")

In [6]:
# Cell 6 â€” Vectorless RAG: Exact rule/constraint checker

def check_compatibility_rules(scenario: str, params: dict) -> list:
    violations = []
    svc      = params.get("service_name", "")
    svc_info = services.get(svc, {})

    if scenario == "framework_upgrade":
        new_ver  = params.get("new_value", "")
        cur_java = int(svc_info.get("java", "8"))
        if new_ver.startswith("3."):
            if cur_java < 17:
                violations.append(f"JAVA_MISMATCH: Spring Boot {new_ver} requires Java >=17, {svc} has Java {cur_java}")
            for b in compatibility_rules["spring_boot_breaking"]["2.7_to_3.x"]:
                violations.append(f"BREAKING_CHANGE: {b}")
        elif new_ver.startswith("2.7") and cur_java < 11:
            violations.append(f"JAVA_MISMATCH: Spring Boot 2.7 requires Java >=11, {svc} has Java {cur_java}")

    elif scenario == "resource_change":
        new_mem  = params.get("new_memory_gb", 0.0)
        peak     = svc_info.get("peak_memory_gb", 0.0)
        restarts = svc_info.get("restarts_30d", 0)
        min_safe = peak * (1 + compatibility_rules["memory_safety_buffer_pct"])
        if new_mem and new_mem < min_safe:
            violations.append(f"MEMORY_UNSAFE: {new_mem}GB < safe minimum {min_safe:.2f}GB (peak {peak}GB + 20% buffer)")
        if restarts >= compatibility_rules["high_restart_threshold"]:
            violations.append(f"HIGH_RESTART_RISK: {svc} had {restarts} restarts in 30d â€” OOMKill likely")

    elif scenario == "db_schema":
        db       = svc_info.get("db", "")
        db_info  = database_users.get(db, {})
        column   = params.get("column", "")
        if len(db_info.get("shared_by", [])) > 1:
            violations.append(f"SHARED_DB: {db} shared by {db_info['shared_by']} â€” all impacted")
        if "NOT NULL" in params.get("constraint", "").upper() and column in db_info.get("has_nullable_cols", []):
            violations.append(f"NULL_VIOLATION: '{column}' has existing NULLs â€” backfill required before constraint")

    elif scenario == "api_contract":
        old_fld  = params.get("old_value", "")
        new_fld  = params.get("new_value", "")
        for api, info in api_consumers.items():
            if info["provider"] == svc:
                if old_fld in info["fields"]:
                    violations.append(f"BREAKING_API: '{old_fld}' -> '{new_fld}' in {api} is not backward compatible")
                if len(info["consumers"]) > 2:
                    violations.append(f"WIDE_IMPACT: {len(info['consumers'])} consumers of {api} will break: {info['consumers']}")

    elif scenario == "shared_dependency":
        lib      = params.get("library_name", "")
        new_ver  = params.get("new_value", "")
        lib_info = libraries.get(lib, {})
        breaking = lib_info.get("breaking_version")
        users    = lib_info.get("used_by", [])
        if breaking and new_ver >= breaking:
            violations.append(f"BREAKING_LIB: {lib} {new_ver} >= breaking version {breaking}")
        if len(users) > 3:
            violations.append(f"BULK_RISK: {len(users)} services use {lib}: {users}")

    elif scenario == "event_schema":
        event      = params.get("event_name", "")
        removed_f  = params.get("removed_field", "")
        event_info = event_consumers.get(event, {})
        for consumer, fields in event_info.get("consumers", {}).items():
            if removed_f in fields:
                crit = services.get(consumer, {}).get("criticality", "unknown")
                violations.append(f"EVENT_FIELD_REMOVED: '{removed_f}' consumed by '{consumer}' ({crit}) â€” will break")

    return violations


print("Vectorless RAG ready")
print(check_compatibility_rules("framework_upgrade", {"service_name": "payment-service", "new_value": "3.2"}))

Vectorless RAG ready
['JAVA_MISMATCH: Spring Boot 3.2 requires Java >=17, payment-service has Java 11', 'BREAKING_CHANGE: javax -> jakarta namespace migration required', 'BREAKING_CHANGE: Removed deprecated Spring APIs', 'BREAKING_CHANGE: spring.factories auto-configuration format removed']


In [7]:
# Cell 7 â€” LangGraph State
from typing import TypedDict

class CGState(TypedDict, total=False):
    change_request:   str
    service_name:     str
    old_value:        str
    new_value:        str
    extra_params:     dict
    change_type:      str    # framework_upgrade|resource_change|db_schema|api_contract|shared_dependency|event_schema
    affected_services: list
    similar_incidents: list
    rule_violations:  list
    memory_lessons:   list
    risk_score:       int    # 0-100
    impact_level:     str    # low|medium|high|critical
    risk_reasons:     list
    rollout_plan:     str
    llm_explanation:  str    # plain-English explanation from local LLM
    llm_remediation:  list   # remediation steps from local LLM
    report:           dict

print("CGState defined")

CGState defined


In [ ]:
# Cell 8 â€” Seven Agent Functions
import re

# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# AGENT 1: Intake
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def intake_agent(state: dict) -> dict:
    req       = state["change_request"]
    req_lower = req.lower()
    extra     = {}

    # Identify service/library
    service_name = "unknown"
    for lib in sorted(libraries, key=len, reverse=True):
        if lib in req_lower:
            service_name = lib
            extra["library_name"] = lib
            break
    if service_name == "unknown":
        for svc in sorted(services, key=len, reverse=True):
            if svc in req_lower:
                service_name = svc
                break

    old_value, new_value = "", ""

    # Memory: "from 2GB to 1GB"
    m = re.search(r'from\s+(\d+(?:\.\d+)?)\s*[Gg][Bb]\s+to\s+(\d+(?:\.\d+)?)\s*[Gg][Bb]', req, re.I)
    if m:
        old_value, new_value = m.group(1), m.group(2)
        extra["new_memory_gb"] = float(new_value)
        extra["old_memory_gb"] = float(old_value)

    # API field: "field from X to Y"
    m = re.search(r'field\s+from\s+(\w+)\s+to\s+(\w+)', req, re.I)
    if m:
        old_value, new_value = m.group(1), m.group(2)

    # Generic version: "from X to Y"
    if not old_value:
        m = re.search(r'from\s+([\w\.]+)\s+to\s+([\w\.]+)', req, re.I)
        if m:
            old_value, new_value = m.group(1), m.group(2)

    # DB NOT NULL column
    m = re.search(r'NOT NULL constraint to\s+([\w\.]+)', req, re.I)
    if m:
        extra["column"] = m.group(1)
        extra["constraint"] = "NOT NULL"
    elif not extra.get("column"):
        m = re.search(r'constraint\s+to\s+([\w\.]+)', req, re.I)
        if m and m.group(1).lower() not in ("not", "null"):
            extra["column"] = m.group(1)
            extra["constraint"] = "NOT NULL"

    # Kafka remove field
    m = re.search(r'[Rr]emove\s+(\w+)\s+field\s+from\s+([a-z][\w-]+)', req)
    if m:
        extra["removed_field"] = m.group(1)
        extra["event_name"]    = m.group(2)

    return {**state, "service_name": service_name, "old_value": old_value,
            "new_value": new_value, "extra_params": extra}


# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# AGENT 2: Scenario Router
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def scenario_router(state: dict) -> dict:
    req   = state["change_request"].lower()
    extra = state.get("extra_params", {})

    if any(k in req for k in ["spring boot", "spring", "framework"]):
        ct = "framework_upgrade"
    elif any(k in req for k in ["memory", "cpu", "limit", "ram", "gb"]):
        ct = "resource_change"
    elif any(k in req for k in ["not null", "schema", "constraint", "column", "migration"]):
        ct = "db_schema"
    elif any(k in req for k in ["kafka", "event", "topic", "stream"]):
        ct = "event_schema"
    elif "library_name" in extra or any(lib in req for lib in libraries):
        ct = "shared_dependency"
    elif any(k in req for k in ["api", "field", "endpoint", "response", "rename"]):
        ct = "api_contract"
    else:
        ct = "framework_upgrade"

    return {**state, "change_type": ct}


# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# AGENT 3: Graph Impact
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def graph_impact_agent(state: dict) -> dict:
    svc   = state.get("service_name", "")
    ct    = state.get("change_type", "")
    extra = state.get("extra_params", {})
    aff   = set()

    if svc in G:
        aff.update(get_affected_services(svc))

    if ct == "shared_dependency":
        lib = extra.get("library_name", svc)
        aff.update(libraries.get(lib, {}).get("used_by", []))

    if ct == "event_schema":
        event = extra.get("event_name", "")
        aff.update(event_consumers.get(event, {}).get("consumers", {}).keys())

    if ct == "api_contract":
        for api, info in api_consumers.items():
            if info["provider"] == svc:
                aff.update(info["consumers"])

    if ct == "db_schema":
        db = services.get(svc, {}).get("db", "")
        aff.update(database_users.get(db, {}).get("shared_by", []))

    aff.discard(svc)
    aff = {s for s in aff if s in services}
    return {**state, "affected_services": sorted(aff)}


# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# AGENT 4: Hybrid RAG
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def hybrid_rag_agent(state: dict) -> dict:
    params = {
        "service_name": state.get("service_name", ""),
        "old_value":    state.get("old_value", ""),
        "new_value":    state.get("new_value", ""),
        **state.get("extra_params", {}),
    }
    return {
        **state,
        "similar_incidents": search_incidents(state["change_request"], k=3),
        "rule_violations":   check_compatibility_rules(state.get("change_type", ""), params),
    }


# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# AGENT 5: Memory Graph
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def memory_graph_agent(state: dict) -> dict:
    svc   = state.get("service_name", "")
    ct    = state.get("change_type", "")
    kws   = {
        "framework_upgrade": ["spring boot", "java", "upgrade", "migrate"],
        "resource_change":   ["memory", "gb", "oom", "resource"],
        "db_schema":         ["not null", "schema", "constraint", "null"],
        "api_contract":      ["api", "field", "rename", "endpoint"],
        "shared_dependency": ["library", "client", "dependency"],
        "event_schema":      ["kafka", "event", "field"],
    }.get(ct, [])
    lessons = [
        {"deployment": m["deployment"], "service": m["service"],
         "change": m["change"], "outcome": m["outcome"], "lessons": m["lessons"]}
        for m in memory_graph
        if m["service"] == svc or any(k in m["change"].lower() for k in kws)
    ]
    return {**state, "memory_lessons": lessons}


# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# AGENT 6: Risk & Rollout (deterministic â€” no LLM, fully auditable)
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def risk_rollout_agent(state: dict) -> dict:
    svc      = state.get("service_name", "")
    ct       = state.get("change_type", "")
    affected = state.get("affected_services", [])
    incs     = state.get("similar_incidents", [])
    viols    = state.get("rule_violations", [])
    mems     = state.get("memory_lessons", [])
    extra    = state.get("extra_params", {})
    svc_info = services.get(svc, {})

    score   = 10
    reasons = ["Base score (+10)"]

    # Criticality
    crit = svc_info.get("criticality", "medium")
    cb   = {"critical": 30, "high": 20, "medium": 10, "low": 5}.get(crit, 10)
    score += cb; reasons.append(f"Service criticality={crit} (+{cb})")

    # Affected services
    ab = min(len(affected) * 5, 25)
    if ab: score += ab; reasons.append(f"{len(affected)} affected service(s) (+{ab})")

    # Incident history
    p1 = [i for i in incs if i.get("severity") == "P1"]
    p2 = [i for i in incs if i.get("severity") == "P2"]
    if p1:   score += 20; reasons.append(f"Similar P1 incident: {p1[0]['id']} (+20)")
    elif p2: score += 10; reasons.append(f"Similar P2 incident: {p2[0]['id']} (+10)")

    # Rule violations
    vb = min(len(viols) * 15, 30)
    if vb: score += vb; reasons.append(f"{len(viols)} rule violation(s) (+{vb})")

    # Memory failures
    if any(m["outcome"] == "failed" for m in mems):
        score += 15; reasons.append("Prior failed deployment for similar change (+15)")

    # Scenario-specific bonuses
    if ct == "framework_upgrade":
        nv = state.get("new_value", "")
        jv = int(svc_info.get("java", "8"))
        if nv.startswith("3.") and jv < 17: score += 20; reasons.append(f"Java mismatch: needs 17, has {jv} (+20)")
        if nv.startswith("3."):             score += 15; reasons.append("Spring Boot 3.x breaking migration (+15)")

    elif ct == "resource_change":
        nm   = extra.get("new_memory_gb", 0.0)
        peak = svc_info.get("peak_memory_gb", 0.0)
        rsts = svc_info.get("restarts_30d", 0)
        if nm and nm < peak * 1.20: score += 25; reasons.append(f"New mem {nm}GB < safe floor {peak*1.2:.2f}GB (+25)")
        if rsts >= 5:                score += 10; reasons.append(f"High restarts: {rsts} in 30d (+10)")

    elif ct == "db_schema":
        db      = svc_info.get("db", "")
        db_info = database_users.get(db, {})
        col     = extra.get("column", "")
        if len(db_info.get("shared_by", [])) > 1: score += 15; reasons.append(f"Shared DB {db} (+15)")
        if col and col in db_info.get("has_nullable_cols", []): score += 20; reasons.append(f"Column '{col}' has NULLs â€” backfill required (+20)")

    elif ct == "api_contract":
        of = state.get("old_value", "")
        for api, info in api_consumers.items():
            if info["provider"] == svc:
                if of and of in info["fields"]: score += 20; reasons.append(f"Breaking field rename '{of}' in {api} (+20)")
                if len(info["consumers"]) > 2:  score += 10; reasons.append(f"{len(info['consumers'])} consumers at risk (+10)")

    elif ct == "shared_dependency":
        lib  = extra.get("library_name", svc)
        nv   = state.get("new_value", "")
        brk  = libraries.get(lib, {}).get("breaking_version")
        if brk and nv and nv >= brk: score += 25; reasons.append(f"Major version break: {lib} {nv} >= {brk} (+25)")

    elif ct == "event_schema":
        event = extra.get("event_name", "")
        rf    = extra.get("removed_field", "")
        for consumer, fields in event_consumers.get(event, {}).get("consumers", {}).items():
            if rf and rf in fields:
                c = services.get(consumer, {}).get("criticality", "")
                if c in ("critical", "high"): score += 20; reasons.append(f"Removed '{rf}' used by {consumer} ({c}) (+20)")

    score = min(score, 100)

    if   score <= 25: impact, rollout, plan = "low",      "direct",         "DIRECT ROLLOUT â€” Low risk. Deploy with standard monitoring."
    elif score <= 50: impact, rollout, plan = "medium",   "canary",         "CANARY ROLLOUT â€” 5% traffic first, monitor 30 min, then 25%->100%."
    elif score <= 75: impact, rollout, plan = "high",     "staged-rollout", "STAGED ROLLOUT â€” Deploy region-by-region with health checks. Rollback plan required."
    else:             impact, rollout, plan = "critical",  "block",          "DEPLOYMENT BLOCKED â€” Resolve all violations before proceeding."

    return {
        **state,
        "risk_score":   score,
        "impact_level": impact,
        "risk_reasons": reasons,
        "rollout_plan": plan,
    }


# AGENT 7: LLM Explanation (Local Qwen/Mistral via Ollama on AMD hardware)
_LLM_SYSTEM = """You are ChangeGuardian AI, a production deployment risk advisor.
You will receive a structured risk analysis. Your job is to:
1. Write a plain-English explanation (3 sentences) of WHY this change is risky.
2. List exactly 3 concrete remediation steps an engineer should take BEFORE deploying.

Respond in this exact JSON format (no markdown fences):
{"explanation": "<3 sentence explanation>", "remediation": ["step 1", "step 2", "step 3"]}"""

def _rule_based_explanation(state: dict) -> tuple:
    """Fallback when LLM is offline â€” generate explanation from rules."""
    ct      = state.get("change_type", "")
    svc     = state.get("service_name", "")
    score   = state.get("risk_score", 0)
    viols   = state.get("rule_violations", [])
    impact  = state.get("impact_level", "unknown")
    inc     = state.get("similar_incidents", [])

    exp = (f"This {ct.replace('_',' ')} change to {svc} carries {impact} risk (score {score}/100). "
           f"{len(viols)} rule violation(s) were detected that must be resolved before deployment. "
           + (f"A similar change previously caused: {inc[0]['title']}." if inc else "No identical past incident found but pattern matches are concerning."))

    remed_map = {
        "framework_upgrade":  ["Run javax->jakarta migration script on entire codebase", "Upgrade Java to 17+ on all pods before deploying", "Run full integration test suite against new Spring Boot version"],
        "resource_change":    ["Verify new memory limit is at least 20% above observed peak usage", "Set up OOMKill alerts before reducing limits", "Perform load test at peak traffic levels first"],
        "db_schema":          ["Backfill all NULL values in the target column before adding constraint", "Run migration in staging with production data copy first", "Coordinate deployment with all services sharing the database"],
        "api_contract":       ["Add backward-compatible alias alongside new field name", "Version the API endpoint (/v2/) before renaming fields", "Notify all consumer teams and agree migration timeline"],
        "shared_dependency":  ["Review CHANGELOG for all breaking changes between old and new version", "Test upgrade in isolation for each consuming service", "Coordinate a single release window across all affected teams"],
        "event_schema":       ["Add new field alongside old field (dual-write) during migration period", "Update all consumers to handle both old and new field names", "Remove old field only after all consumers confirm migration"],
    }
    remed = remed_map.get(ct, ["Review all rule violations", "Test in staging first", "Have rollback plan ready"])
    return exp, remed


def llm_explanation_agent(state: dict) -> dict:
    """
    Calls local Ollama LLM (Qwen2.5/Mistral) running on AMD hardware.
    The LLM receives the full deterministic risk analysis and generates:
      - Plain-English explanation (3 sentences)
      - 3 concrete remediation steps
    Falls back to rule-based text if Ollama is not running.
    """
    if not OLLAMA_READY:
        explanation, remediation = _rule_based_explanation(state)
    else:
        prompt = json.dumps({
            "change_request":   state.get("change_request"),
            "service":          state.get("service_name"),
            "scenario":         state.get("change_type"),
            "risk_score":       state.get("risk_score"),
            "impact_level":     state.get("impact_level"),
            "affected_services":state.get("affected_services", []),
            "rule_violations":  state.get("rule_violations", []),
            "similar_incidents":[{"id":i["id"],"title":i["title"],"severity":i["severity"]} for i in state.get("similar_incidents",[])],
            "risk_reasons":     state.get("risk_reasons", []),
            "rollout_plan":     state.get("rollout_plan", ""),
        }, indent=2)
        raw = call_llm(prompt, system=_LLM_SYSTEM)
        try:
            parsed      = json.loads(raw)
            explanation = parsed.get("explanation", raw)
            remediation = parsed.get("remediation", [])
        except Exception:
            explanation, remediation = _rule_based_explanation(state)

    report = {
        "change_request":   state.get("change_request"),
        "service":          state.get("service_name"),
        "scenario":         state.get("change_type"),
        "affected_services":state.get("affected_services", []),
        "similar_incidents":[{"id":i["id"],"title":i["title"],"severity":i["severity"]} for i in state.get("similar_incidents",[])],
        "rule_violations":  state.get("rule_violations", []),
        "memory_lessons":   [{"deployment":m["deployment"],"outcome":m["outcome"],"lessons":m["lessons"]} for m in state.get("memory_lessons",[])],
        "risk_score":       state.get("risk_score"),
        "impact_level":     state.get("impact_level"),
        "risk_reasons":     state.get("risk_reasons", []),
        "rollout_plan":     state.get("rollout_plan"),
        "llm_explanation":  explanation,
        "llm_remediation":  remediation,
    }

    return {
        **state,
        "llm_explanation": explanation,
        "llm_remediation": remediation,
        "report":          report,
    }


print("All 7 agent functions defined")

All 7 agent functions defined


In [ ]:
# Cell 9 â€” Build LangGraph Workflow (7 nodes)
from langgraph.graph import StateGraph, END

def build_workflow():
    builder = StateGraph(dict)

    builder.add_node("intake",       intake_agent)
    builder.add_node("router",       scenario_router)
    builder.add_node("graph_impact", graph_impact_agent)
    builder.add_node("hybrid_rag",   hybrid_rag_agent)
    builder.add_node("memory_graph", memory_graph_agent)
    builder.add_node("risk_rollout", risk_rollout_agent)
    builder.add_node("llm_explain",  llm_explanation_agent)  

    builder.set_entry_point("intake")
    builder.add_edge("intake",       "router")
    builder.add_edge("router",       "graph_impact")
    builder.add_edge("graph_impact", "hybrid_rag")
    builder.add_edge("hybrid_rag",   "memory_graph")
    builder.add_edge("memory_graph", "risk_rollout")
    builder.add_edge("risk_rollout", "llm_explain")
    builder.add_edge("llm_explain",  END)

    return builder.compile()


workflow = build_workflow()
print("LangGraph workflow compiled")
print("intake -> router -> graph_impact -> hybrid_rag -> memory_graph -> risk_rollout -> llm_explain -> END")

LangGraph workflow compiled
intake -> router -> graph_impact -> hybrid_rag -> memory_graph -> risk_rollout -> llm_explain -> END


In [18]:
# Cell 10 â€” Run 6 Demo Scenarios

ICONS = {"low": "", "medium": "", "high": " ", "critical": ""}

def print_report(result: dict):
    r      = result.get("report", {})
    impact = r.get("impact_level", "unknown")
    score  = r.get("risk_score", 0)
    icon   = ICONS.get(impact, "âšª")

    print("=" * 72)
    print(f"  CHANGE   : {r.get('change_request')}")
    print(f"  SCENARIO : {r.get('scenario','').replace('_',' ').upper()}")
    print(f"  SERVICE  : {r.get('service')}")
    print("-" * 72)
    aff = r.get("affected_services", [])
    print(f"  AFFECTED ({len(aff)}): {', '.join(aff) or 'None'}")

    print("\n  SIMILAR INCIDENTS (Vector RAG):")
    for i in r.get("similar_incidents", []):
        print(f"    [{i['severity']}] {i['id']}: {i['title']}")

    print("\n  RULE VIOLATIONS (Vectorless RAG):")
    for v in r.get("rule_violations", []) or ["  None"]:
        print(f"    ! {v}")

    print("\n  MEMORY GRAPH:")
    for m in r.get("memory_lessons", []) or [{"outcome":"none","deployment":"-","lessons":["No prior data"]}]:
        tag = "FAILED" if m["outcome"] == "failed" else "SUCCESS" if m["outcome"] == "success" else "-"
        print(f"    [{m['deployment']}][{tag}] {' | '.join(m['lessons'])}")

    print("\n  RISK BREAKDOWN:")
    for rr in r.get("risk_reasons", []):
        print(f"    + {rr}")

    print(f"\n  {icon} RISK SCORE: {score}/100  |  IMPACT: {impact.upper()}")
    print(f"  ROLLOUT: {r.get('rollout_plan')}")

    print("\n  LLM EXPLANATION (AMD Local LLM):")
    print(f"    {r.get('llm_explanation','')}")

    print("\n  LLM REMEDIATION STEPS:")
    for j, step in enumerate(r.get("llm_remediation", []), 1):
        print(f"    {j}. {step}")

    print("=" * 72)
    print()


scenarios = [
    "Upgrade payment-service from Spring Boot 2.7 to 3.2",
    "Reduce checkout-service memory limit from 2GB to 1GB",
    "Add NOT NULL constraint to transaction.amount in payment-db",
    "Change payment API response field from customer_id to customerId",
    "Upgrade legacy-auth-client from 1.4.0 to 2.0.0 across all services",
    "Remove customerEmail field from order-created Kafka event",
]

ram_before = psutil.virtual_memory().used / (1024 ** 3)
print(f"RAM before run: {ram_before:.1f} GB used / {ram_gb:.1f} GB total\n")

all_results = []
for i, sc in enumerate(scenarios, 1):
    print(f"[{i}/6] Running: {sc}")
    t0 = time.time()
    result = workflow.invoke({"change_request": sc})
    elapsed = time.time() - t0
    all_results.append(result)
    print_report(result)
    print(f"  (pipeline took {elapsed:.1f}s | RAM: {psutil.virtual_memory().used/(1024**3):.1f} GB)\n")

print("\n" + "=" * 72)
print("  SUMMARY TABLE")
print("=" * 72)
print(f"{'#':<3} {'Score':>5} {'Impact':<10} {'Scenario':<30} {'Rollout'}")
print("-" * 72)
for i, r in enumerate(all_results, 1):
    rpt = r.get("report", {})
    icon = ICONS.get(rpt.get("impact_level",""),"")
    print(f"{i:<3} {rpt.get('risk_score',0):>5}  {icon}{rpt.get('impact_level',''):<9} {rpt.get('scenario',''):<30} {rpt.get('rollout_plan','')[:35]}")
print("=" * 72)

RAM before run: 11.3 GB used / 15.4 GB total

[1/6] Running: Upgrade payment-service from Spring Boot 2.7 to 3.2
  [LLM] 2.1s | RAM in use: 11.3/15.4 GB
  CHANGE   : Upgrade payment-service from Spring Boot 2.7 to 3.2
  SCENARIO : FRAMEWORK UPGRADE
  SERVICE  : payment-service
------------------------------------------------------------------------
  AFFECTED (5): checkout-service, inventory-service, notification-svc, order-service, user-service

  SIMILAR INCIDENTS (Vector RAG):
    [P1] INC-002: Spring Boot 3 migration broke javax imports
    [P1] INC-005: legacy-auth-client 2.0 removed token refresh method
    [P2] INC-004: API field rename broke checkout

  RULE VIOLATIONS (Vectorless RAG):
    !   None

  MEMORY GRAPH:
    [DEP-101][SUCCESS] Ran full regression suite | Deployed on low-traffic Sunday

  RISK BREAKDOWN:
    + Base score (+10)
    + Service criticality=critical (+30)
    + 5 affected service(s) (+25)
    + Similar P1 incident: INC-002 (+20)

   RISK SCORE: 85/100  | 

In [ ]:
import gradio as gr

def analyze_change(change_request: str) -> str:
    if not change_request.strip():
        return "Please enter a change request."

    t0     = time.time()
    result = workflow.invoke({"change_request": change_request.strip()})
    elapsed = time.time() - t0
    ram_now = psutil.virtual_memory().used / (1024**3)
    r      = result.get("report", {})

    impact = r.get("impact_level", "unknown")
    score  = r.get("risk_score", 0)
    icon   = ICONS.get(impact)

    aff_md   = ", ".join(f"`{s}`" for s in r.get("affected_services",[])) or "_None_"
    viols_md = "\n".join(f"- `{v}`" for v in r.get("rule_violations",[])) or "_No violations_"
    incs_md  = "\n".join(f"- **[{i['severity']}]** `{i['id']}`: {i['title']}" for i in r.get("similar_incidents",[])) or "_None_"
    mems_md  = "\n".join(
        f"- {'**FAILED**' if m['outcome']=='failed' else '**SUCCESS**'} `{m['deployment']}`: {' | '.join(m['lessons'])}"
        for m in r.get("memory_lessons",[])
    ) or "_No prior deployments_"
    reasons_md = "\n".join(f"- {rr}" for rr in r.get("risk_reasons",[])) or "_None_"
    remed_md   = "\n".join(f"{j}. {s}" for j,s in enumerate(r.get("llm_remediation",[]),1)) or "_None_"

    return rf"""## {icon} ChangeGuardian AI  Risk Report

| | |
|---|---|
| **Change** | {r.get('change_request')} |
| **Scenario** | `{r.get('scenario','').replace('_',' ').upper()}` |
| **Service** | `{r.get('service')}` |
| **Risk Score** | **{score} / 100** |
| **Impact** | {icon} **{impact.upper()}** |
| **Rollout** | {r.get('rollout_plan')} |
| **Pipeline time** | {elapsed:.1f}s \| AMD RAM in use: {ram_now:.1f}/{ram_gb:.1f} GB |

---

###  Affected Services
{aff_md}

###  Rule Violations (Vectorless RAG)
{viols_md}

###  Similar Past Incidents (Vector RAG  FAISS)
{incs_md}

###  Memory Graph
{mems_md}

###  Risk Score Breakdown
{reasons_md}

---

###  LLM Explanation  *(Local {OLLAMA_MODEL} " {'AMD GPU/ROCm' if OLLAMA_READY else 'rule-based fallback'})*
{r.get('llm_explanation','')}

###  Remediation Steps
{remed_md}
"""


demo = gr.Interface(
    fn=analyze_change,
    inputs=gr.Textbox(
        label="Describe your production change",
        placeholder="e.g. Upgrade payment-service from Spring Boot 2.7 to 3.2",
        lines=3,
    ),
    outputs=gr.Markdown(label="Risk Analysis Report"),
    title="ChangeGuardian AI",
    description=(
        f"Pre-deployment production risk analysis \n"
        f"Running on AMD hardware . RAM: {ram_gb:.0f} GB · CPU: {cpu_cores} threads · "
        f"LLM: {OLLAMA_MODEL} ({'LIVE' if OLLAMA_READY else 'offline-fallback'})"
    ),
    examples=[
        ["Upgrade payment-service from Spring Boot 2.7 to 3.2"],
        ["Reduce checkout-service memory limit from 2GB to 1GB"],
        ["Add NOT NULL constraint to transaction.amount in payment-db"],
        ["Change payment API response field from customer_id to customerId"],
        ["Upgrade legacy-auth-client from 1.4.0 to 2.0.0 across all services"],
        ["Remove customerEmail field from order-created Kafka event"],
    ],
)

demo.launch(share=True, theme=gr.themes.Soft())